# 04 — Engenharia de Features (Motor Geométrico)

Transforma os arquivos `.json` de detecção em um **vetor numérico fixo de 107 colunas**, pronto para treinar o Random Forest no Módulo 5.

**Como funciona:**
1. **NMS:** Remove detecções duplicadas da mesma bola física antes de qualquer cálculo.
2. **Cálculos geométricos:** Mede distâncias e ângulos entre branca, bolas-alvo e caçapas.
3. **Normalização:** Converte todas as métricas para proporções (valores entre -1 e 1).
4. **Zero-padding:** Garante que o vetor final tenha sempre exatamente 107 colunas.

**Entrada:** `data/output/*_balls.json`  
**Saída:** `data/dataset.csv`

**Codificação de tipo:**

| Valor | Tipo |
|---|---|
| `1.0` | lisa |
| `-1.0` | listrada |
| `2.0` | preta |
| `0.0` | vazio (zero-padding) |

## Célula 1 — Imports e Caminhos

In [1]:
import os
import glob
import json
import numpy as np
import pandas as pd

CAMINHO_OUTPUT = os.path.join('..', 'data', 'output')
CAMINHO_CSV    = os.path.join('..', 'data', 'dataset.csv')

IMG_W, IMG_H = 800.0, 400.0  

## Célula 2 — Motor Geométrico

A classe `MotorGeometrico` encapsula toda a matemática do jogo:

- **`calc_distancia`**: distância euclidiana entre dois pontos.
- **`calc_angulo`**: direção da trajetória via `arctan2`.
- **`calc_angulo_corte`**: dificuldade da jogada — diferença angular entre o ângulo de ataque (branca → bola) e o ângulo ideal (bola → caçapa mais próxima).
- **`processar_estado`**: extrai features de cada bola e monta o vetor de 107 valores.

In [2]:
class MotorGeometrico:
    # Mapeamento tipo string → valor numérico (deve bater com o Módulo 5)
    TIPO_NUM = {'lisa': 1.0, 'listrada': -1.0, 'preta': 2.0}

    def __init__(self, largura=800.0, altura=400.0):
        self.largura  = largura
        self.altura   = altura
        self.diag_max = np.hypot(largura, altura)

    def calc_distancia(self, p1, p2):
        """Distância euclidiana entre dois pontos (x, y)."""
        
        return np.hypot(p2[0] - p1[0], p2[1] - p1[1])

    def calc_angulo(self, p1, p2):
        """Ângulo da trajetória p1 → p2 em radianos [-π, π]."""

        return np.arctan2(p2[1] - p1[1], p2[0] - p1[0])

    def calc_angulo_corte(self, ang_branca_bola, ang_bola_cacapa):
        """Dificuldade da jogada: diferença angular entre ataque e caçapa."""

        diff = (ang_bola_cacapa - ang_branca_bola + np.pi) % (2 * np.pi) - np.pi
        return abs(diff)

    def processar_estado(self, dados_json, limiar_nms=25):
        """
        Converte um JSON de detecção num vetor numpy de 107 features.
        """

        cacapas   = [(c['x'], c['y']) for c in dados_json.get('cacapas', [])]
        bolas = dados_json.get('bolas', [])

        branca      = next((b for b in bolas if b['tipo'] == 'branca'), None)
        outras_bolas = [b for b in bolas if b['tipo'] != 'branca']

        if not branca:
            return np.zeros(107)

        p_branca        = (branca['x'], branca['y'])
        features_branca = [branca['x'] / self.largura, branca['y'] / self.altura]

        lista_features_bolas = []

        for bola in outras_bolas:
            p_bola = (bola['x'], bola['y'])

            # Codificação correta: lisa=1.0, listrada=-1.0, preta=2.0
            tipo_num = self.TIPO_NUM.get(bola['tipo'], 0.0)

            dist_w_b = self.calc_distancia(p_branca, p_bola)
            ang_w_b  = self.calc_angulo(p_branca, p_bola)

            dists_cacapas = [self.calc_distancia(p_bola, c) for c in cacapas]
            idx_cac_prox  = int(np.argmin(dists_cacapas))
            p_cacapa_prox = cacapas[idx_cac_prox]
            dist_b_p      = dists_cacapas[idx_cac_prox]
            ang_b_p       = self.calc_angulo(p_bola, p_cacapa_prox)
            ang_corte     = self.calc_angulo_corte(ang_w_b, ang_b_p)

            features_bola = [
                tipo_num,
                bola['x'] / self.largura,
                bola['y'] / self.altura,
                dist_w_b  / self.diag_max,
                ang_w_b   / np.pi,
                dist_b_p  / self.diag_max,
                ang_corte / np.pi,
            ]
            lista_features_bolas.append((dist_w_b, features_bola))

        lista_features_bolas.sort(key=lambda x: x[0])
        vetor_bolas = [f[1] for f in lista_features_bolas]

        vetor_bolas = vetor_bolas[:15]
        while len(vetor_bolas) < 15:
            vetor_bolas.append([0.0] * 7)

        return np.concatenate(([features_branca], vetor_bolas), axis=None)


print('MotorGeometrico carregado.')
print(f'Codificação: {MotorGeometrico.TIPO_NUM}')

MotorGeometrico carregado.
Codificação: {'lisa': 1.0, 'listrada': -1.0, 'preta': 2.0}


## Célula 3 — Processamento de Um Único JSON (Teste)

Útil para inspecionar o vetor gerado antes de rodar o batch completo.

Altere `NOME_JSON` para testar outro arquivo.

In [4]:
NOME_JSON = 'IMG_1034_balls.json'

caminho_json = os.path.join(CAMINHO_OUTPUT, NOME_JSON)

try:
    with open(caminho_json, 'r', encoding='utf-8') as f:
        dados_da_mesa = json.load(f)
    print(f'JSON carregado: {NOME_JSON}')
    print(f'  Bolas brutas : {len(dados_da_mesa['bolas'])}')
except FileNotFoundError:
    print(f'[ERRO] {caminho_json} não encontrado.')
    dados_da_mesa = None

if dados_da_mesa:
    motor = MotorGeometrico()
    vetor = motor.processar_estado(dados_da_mesa)

    colunas = ['branca_x', 'branca_y']
    for i in range(1, 16):
        colunas.extend([f'b{i}_tipo', f'b{i}_x', f'b{i}_y',
                        f'b{i}_dist_branca', f'b{i}_ang_branca',
                        f'b{i}_dist_cacapa', f'b{i}_ang_corte'])

    df_teste = pd.DataFrame([vetor], columns=colunas)

    if not os.path.exists(CAMINHO_CSV):
        df_teste.to_csv(CAMINHO_CSV, index=False)
        print(f'Dataset criado: {CAMINHO_CSV}')
    else:
        df_teste.to_csv(CAMINHO_CSV, mode='a', header=False, index=False)
        print(f'Linha adicionada: {CAMINHO_CSV}')

    print(f'Vetor: {vetor.shape[0]} colunas')
    print(df_teste.T.rename(columns={0: 'valor'}).head(20))

JSON carregado: IMG_1034_balls.json
  Bolas brutas : 16
Dataset criado: ../data/dataset.csv
Vetor: 107 colunas
                   valor
branca_x        0.047500
branca_y        0.627500
b1_tipo         1.000000
b1_x            0.085000
b1_y            0.567500
b1_dist_branca  0.042953
b1_ang_branca  -0.214777
b1_dist_cacapa  0.184909
b1_ang_corte    0.816584
b2_tipo        -1.000000
b2_x            0.066250
b2_y            0.875000
b2_dist_branca  0.111949
b2_ang_branca   0.452135
b2_dist_cacapa  0.056181
b2_ang_corte    0.311306
b3_tipo        -1.000000
b3_x            0.213750
b3_y            0.267500
b3_dist_branca  0.219160


## Célula 4 — Batch: Reconstrói o Dataset Completo

Processa **todos** os `_balls.json` em `data/output/` e regera o `dataset.csv` do zero.

Execute esta célula em vez da anterior para gerar o dataset de treinamento completo. Frames sem bola branca detectada são ignorados e listados ao final.

In [6]:
arquivos_json = sorted(glob.glob(os.path.join(CAMINHO_OUTPUT, '*_balls.json')))
print(f'JSONs encontrados: {len(arquivos_json)}')

if not arquivos_json:
    print('Nenhum JSON encontrado. Rode o 04_detector_bolas.ipynb primeiro.')
else:
    colunas = ['branca_x', 'branca_y']
    for i in range(1, 16):
        colunas.extend([f'b{i}_tipo', f'b{i}_x', f'b{i}_y',
                        f'b{i}_dist_branca', f'b{i}_ang_branca',
                        f'b{i}_dist_cacapa', f'b{i}_ang_corte'])

    motor  = MotorGeometrico()
    linhas = []
    erros  = []

    for caminho_json in arquivos_json:
        nome = os.path.basename(caminho_json)
        try:
            with open(caminho_json, 'r', encoding='utf-8') as f:
                dados = json.load(f)

            bolas = dados.get('bolas', [])
            tem_branca   = any(b['tipo'] == 'branca' for b in bolas)

            if not tem_branca:
                print(f'[IGNORADO] {nome} — sem bola branca detectada')
                erros.append(nome)
                continue

            dados_nms = dict(dados)
            dados_nms['bolas'] = bolas
            vetor = motor.processar_estado(dados_nms, limiar_nms=0)
            linhas.append(vetor)
            print(f'{nome} ({len(bolas)} bolas.')

        except Exception as e:
            print(f'[ERRO] {nome}: {e}')
            erros.append(nome)

    if linhas:
        df = pd.DataFrame(linhas, columns=colunas)
        df.to_csv(CAMINHO_CSV, index=False)
        print(f'\nDataset reconstruído: {len(linhas)} linhas → {CAMINHO_CSV}')
        if erros:
            print(f'Frames ignorados/com erro ({len(erros)}): {erros}')
    else:
        print('Nenhuma linha válida gerada. Verifique a detecção.')

JSONs encontrados: 50
IMG_0641_balls.json (16 bolas.
IMG_0642_balls.json (16 bolas.
IMG_0643_balls.json (15 bolas.
IMG_0645_balls.json (15 bolas.
IMG_0646_balls.json (15 bolas.
IMG_0647_balls.json (3 bolas.
IMG_0649_balls.json (3 bolas.
IMG_0650_balls.json (3 bolas.
IMG_0652_balls.json (3 bolas.
IMG_1032_balls.json (16 bolas.
IMG_1033_balls.json (16 bolas.
IMG_1034_balls.json (16 bolas.
IMG_1035_balls.json (16 bolas.
IMG_1036_balls.json (14 bolas.
IMG_1037_balls.json (15 bolas.
IMG_1038_balls.json (15 bolas.
IMG_1039_balls.json (15 bolas.
IMG_1040_balls.json (15 bolas.
IMG_1041_balls.json (15 bolas.
IMG_1042_balls.json (15 bolas.
IMG_1043_balls.json (15 bolas.
IMG_1044_balls.json (15 bolas.
IMG_1045_balls.json (14 bolas.
IMG_1046_balls.json (12 bolas.
IMG_1047_balls.json (11 bolas.
IMG_1048_balls.json (11 bolas.
IMG_1049_balls.json (10 bolas.
IMG_1050_balls.json (10 bolas.
IMG_1051_balls.json (10 bolas.
IMG_1052_balls.json (15 bolas.
IMG_1053_balls.json (15 bolas.
IMG_1054_balls.json (